In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Přesná a zašuměná simulace s primitivy Qiskit Aer

<details>
<summary><b>Verze balíčků</b></summary>

Kód na této stránce byl vyvinut s použitím následujících požadavků.
Doporučujeme používat tyto verze nebo novější.

```
qiskit[all]~=2.3.0
qiskit-aer~=0.17
```
</details>
[Přesná simulace s primitivy Qiskit](/guides/simulate-with-qiskit-sdk-primitives) ukazuje, jak používat referenční primitivy obsažené v Qiski k provádění přesné simulace kvantových Circuit. Stávající kvantové procesory trpí chybami, neboli šumem, takže výsledky přesné simulace nemusí nutně odpovídat výsledkům, které bys očekával(a) při spuštění Circuit na skutečném hardwaru. Zatímco referenční primitivy v Qiski modelování šumu nepodporují, [Qiskit Aer](https://qiskit.org/ecosystem/aer/) obsahuje implementace primitiv, které modelování šumu podporují. Qiskit Aer je vysoce výkonný simulátor kvantových Circuit, který můžeš použít místo referenčních primitiv pro lepší výkon a více funkcí. Je součástí [ekosystému Qiskit](https://qiskit.github.io/ecosystem/). V tomto článku ukážeme použití primitiv Qiskit Aer pro přesnou a zašuměnou simulaci.

> **Note:** - Je vyžadován `qiskit-aer` v0.14 nebo novější.
> - Přestože primitivy Qiskit Aer implementují rozhraní primitiv, neposkytují stejné možnosti jako primitivy Qiskit Runtime. Například úroveň odolnosti (resilience level) není u primitiv Qiskit Aer dostupná.
> - Podrobnosti o možnostech simulační metody, které Aer podporuje, najdeš v [dokumentaci AerSimulator](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator).

Pro prozkoumání přesné a zašuměné simulace vytvoř ukázkový Circuit na osmi Qubitech:

In [1]:
from qiskit.circuit.library import efficient_su2

n_qubits = 8
circuit = efficient_su2(n_qubits)
circuit.draw("mpl")

<Image src="../docs/images/guides/simulate-with-qiskit-aer/extracted-outputs/df70b5fd-971d-4e7d-a23a-8df037c0fa47-0.svg" alt="Output of the previous code cell" />

![Výstup předchozí buňky kódu](../docs/images/guides/simulate-with-qiskit-aer/extracted-outputs/df70b5fd-971d-4e7d-a23a-8df037c0fa47-0.svg)

Tento Circuit obsahuje parametry reprezentující rotační úhly pro Gate $R_y$ a $R_z$. Při simulaci tohoto Circuit musíme zadat explicitní hodnoty těchto parametrů. V další buňce zadáme hodnoty těchto parametrů a použijeme primitiv Estimator z Qiskit Aer k výpočtu přesné střední hodnoty observablu $ZZ \cdots Z$.

In [2]:
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Estimator

observable = SparsePauliOp("Z" * n_qubits)
params = [0.1] * circuit.num_parameters

exact_estimator = Estimator()
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(circuit)
pub = (isa_circuit, observable, params)
job = exact_estimator.run([pub])
result = job.result()
pub_result = result[0]
exact_value = float(pub_result.data.evs)
exact_value

0.8870140234256602

Now, let's initialize a noise model that includes depolarizing error of 2% on every CX gate. In practice, the error arising from the two-qubit gates, which are CX gates here, are the dominant source of error when running a circuit. See [Build noise models](./build-noise-models) for an overview of constructing noise models in Qiskit Aer.

In the next cell, we construct an Estimator that incorporates this noise model and use it to compute the expectation value of the observable.

In [3]:
from qiskit_aer.noise import NoiseModel, depolarizing_error

noise_model = NoiseModel()
cx_depolarizing_prob = 0.02
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(cx_depolarizing_prob, 2), ["cx"]
)

noisy_estimator = Estimator(
    options=dict(backend_options=dict(noise_model=noise_model))
)
job = noisy_estimator.run([pub])
result = job.result()
pub_result = result[0]
noisy_value = float(pub_result.data.evs)
noisy_value

0.7247404214143528

Nyní inicializujeme model šumu, který zahrnuje depolarizační chybu 2 % na každém Gate CX. V praxi jsou chyby pocházející z dvoQubitových Gate, v tomto případě Gate CX, dominantním zdrojem chyb při spouštění Circuit. Přehled vytváření modelů šumu v Qiskit Aer najdeš v části [Sestavení modelů šumu](./build-noise-models).

V další buňce sestavíme Estimator zahrnující tento model šumu a použijeme ho k výpočtu střední hodnoty observablu.

In [4]:
cx_count = circuit.count_ops()["cx"]
(1 - cx_depolarizing_prob) ** cx_count

0.6542558123199923

This value, 65%, gives a rough estimate of the probability that our final state is correct. It is a conservative estimate because it does not take into account the initial state of the simulation.

The following code cell shows how to use the Sampler primitive from Qiskit Aer to sample from the noisy circuit. We need to add measurements to the circuit before running it with the Sampler primitive.

In [5]:
from qiskit_aer.primitives import SamplerV2 as Sampler

measured_circuit = circuit.copy()
measured_circuit.measure_all()

noisy_sampler = Sampler(
    options=dict(backend_options=dict(noise_model=noise_model))
)
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(measured_circuit)
pub = (isa_circuit, params, 100)
job = noisy_sampler.run([pub])
result = job.result()
pub_result = result[0]
pub_result.data.meas.get_counts()

{'00100000': 1,
 '00000000': 65,
 '10101000': 1,
 '10000000': 5,
 '00001000': 1,
 '00000110': 2,
 '11110010': 1,
 '00000011': 3,
 '01010000': 3,
 '11000000': 3,
 '01111000': 1,
 '01000000': 2,
 '00000010': 1,
 '01100000': 1,
 '00011000': 1,
 '00111100': 1,
 '00010100': 1,
 '00001111': 1,
 '00110000': 1,
 '01100101': 1,
 '00000100': 1,
 '10100000': 1,
 '00000001': 1,
 '11010000': 1}

Jak vidíš, střední hodnota v přítomnosti šumu je poměrně daleko od správné hodnoty. V praxi můžeš využít různé techniky potlačení chyb k boji proti efektům šumu, ale diskuze o těchto technikách je nad rámec tohoto článku.

Abychom získali hrubou představu o tom, jak šum ovlivňuje konečný výsledek, uvažujme náš model šumu, který přidává depolarizační chybu 2 % ke každému Gate CX. Depolarizační chyba s pravděpodobností $p$ je definována jako kvantový kanál $E$, který má následující působení na matici hustoty $\rho$:

$$
E(\rho) = (1 - p) \rho + p\frac{I}{2^n}
$$

kde $n$ je počet Qubitů, v tomto případě 2. To znamená, že s pravděpodobností $p$ je stav nahrazen zcela smíšeným stavem a stav je zachován s pravděpodobností $1 - p$. Po $m$ aplikacích depolarizačního kanálu by pravděpodobnost zachování stavu byla $(1 - p)^m$. Proto očekáváme, že pravděpodobnost zachování správného stavu na konci simulace klesá exponenciálně s počtem Gate CX v našem Circuit.

Spočítejme počet Gate CX v našem Circuit a vypočítejme $(1 - p)^m$. Zavoláme `count_ops`, abychom získali slovník mapující názvy Gate na počty, a načteme záznam pro Gate CX.